# ConnectX: Minimax Agent with Alpha-Beta Pruning

**Simulation Competition -- Connect Four**

**Author:** [Amey Thakur](https://www.kaggle.com/ameythakur20)

This notebook builds a ConnectX agent from scratch. The agent uses
minimax tree search with alpha-beta pruning, center-first move
ordering, and a hand-tuned board evaluation function. Every design
choice is explained in detail so the reasoning behind each heuristic
is clear, and each code block is written to be readable by beginners.

**Outline:**

1. Environment setup and board representation
2. Game rules and strategy overview
3. Board evaluation heuristics
4. Minimax search with alpha-beta pruning
5. Agent assembly and local testing
6. Performance evaluation against built-in agents
7. Submission file generation and validation

**Scoring:** ELO-style skill rating based on head-to-head matches.

---

## 1. Environment Setup

The `kaggle-environments` package provides the ConnectX game engine.
It handles board state, turn management, win detection, and rendering.
It is pre-installed in the Kaggle environment so we simply import it
and create a game instance to inspect the default configuration:
7 columns, 6 rows, 4 in a row to win.

In [ ]:
from kaggle_environments import evaluate, make, utils
import os

# create the ConnectX environment with debug output enabled
env = make('connectx', debug=True)

# print the default game configuration
print('Columns:', env.configuration.columns)
print('Rows:   ', env.configuration.rows)
print('In-a-Row:', env.configuration.inarow)

## 2. Board Representation

The game board is stored as a flat list of 42 integers (6 rows times
7 columns). Each cell is either 0 (empty), 1 (player 1), or 2
(player 2). The list is ordered row by row from the top-left corner:
index 0 is the top-left cell, index 6 is the top-right cell,
index 7 is the start of the second row, and so on.

For the agent, we work directly with this flat list rather than
reshaping into a 2D array. Flat indexing is faster and avoids
importing NumPy, which matters inside the 2-second time limit.

In [ ]:
# visualize the board index layout
rows, cols = 6, 7
print('Board index layout (row-major order):')
print()
for r in range(rows):
    print('  '.join(f'{r * cols + c:2d}' for c in range(cols)))
print()
print(f'Total cells: {rows * cols}')

## 3. Game Rules

A quick recap of the rules, since the agent logic is built directly
around them:

- Two players alternate turns. Player 1 moves first.
- On each turn, a player drops a checker into one of the 7 columns.
  The checker falls to the lowest empty row in that column.
- The first player to get 4 checkers in a row (horizontally,
  vertically, or diagonally) wins.
- If all 42 cells are filled with no winner, the game is a draw.
- Dropping into a full column or returning an invalid column index
  results in an automatic loss.
- Each move must be returned within 2 seconds (60 seconds on the
  very first move of the game).

## 4. Strategy Overview

The agent is built around three ideas that have proven effective in
Connect Four competitions on Kaggle:

**Minimax with alpha-beta pruning.** Minimax is a decision-making
algorithm for two-player zero-sum games. It assumes the opponent plays
optimally and searches the game tree to a fixed depth. Alpha-beta
pruning cuts branches that cannot influence the final decision,
allowing the search to go deeper within the same time budget.

**Move ordering.** Alpha-beta pruning works best when the strongest
moves are examined first. We order columns from the center outward
(3, 2, 4, 1, 5, 0, 6) because center columns give more connection
opportunities. Within that, immediate wins and blocks are checked
before entering the full search.

**Board evaluation heuristic.** When the search reaches its depth
limit, a heuristic function estimates how favorable the board is.
The heuristic scores every window of 4 consecutive cells on the
board (horizontal, vertical, and both diagonal directions) and
combines those scores into a single number.

## 5. Building the Agent

The entire agent is written as a single self-contained function
called `agent`. All helper functions are defined inside it so that
the submission file has no external dependencies. This is a Kaggle
requirement: the submitted `.py` file must be fully encapsulated.

We build the agent piece by piece below, then assemble everything
into the final function.

### 5.1 Dropping a Checker and Finding Valid Columns

Two small helpers: one checks which columns still have room, and
the other simulates dropping a checker into a column by finding
the lowest empty row.

In [ ]:
def get_valid_columns(board, cols):
    """Return a list of column indices that are not full."""
    return [c for c in range(cols) if board[c] == 0]


def drop_checker(board, col, mark, rows, cols):
    """Drop a checker into the given column and return the new board.
    The checker lands on the lowest empty row in that column."""
    new_board = list(board)  # shallow copy is enough for a flat list of ints
    for row in range(rows - 1, -1, -1):
        idx = row * cols + col
        if new_board[idx] == 0:
            new_board[idx] = mark
            return new_board
    return new_board  # column was full (should not happen if validated)


# quick test: drop a checker into column 3 on an empty board
test_board = [0] * 42
test_board = drop_checker(test_board, 3, 1, 6, 7)
print('Checker landed at index:', test_board.index(1))  # should be 38 (bottom row, col 3)

### 5.2 Win Detection

After dropping a checker, we need to know if it created four in a
row. Rather than scanning the entire board, we only check the four
directions from the cell where the checker was placed: horizontal,
vertical, and both diagonals. This is faster because Connect Four
wins always pass through the most recently placed piece.

In [ ]:
def is_win(board, col, mark, rows, cols, inarow):
    """Check whether dropping into a given column would create a win for mark.
    Assumes the checker has NOT been dropped yet -- it figures out where
    the checker would land and checks from that position."""
    # find the row where the checker would land
    row = -1
    for r in range(rows - 1, -1, -1):
        if board[r * cols + col] == 0:
            row = r
            break
    if row == -1:
        return False  # column is full

    # directions to check: (row_delta, col_delta)
    directions = [(0, 1), (1, 0), (1, 1), (1, -1)]

    for dr, dc in directions:
        count = 1  # count the placed piece itself
        # count in the positive direction
        for i in range(1, inarow):
            nr, nc = row + dr * i, col + dc * i
            if 0 <= nr < rows and 0 <= nc < cols and board[nr * cols + nc] == mark:
                count += 1
            else:
                break
        # count in the negative direction
        for i in range(1, inarow):
            nr, nc = row - dr * i, col - dc * i
            if 0 <= nr < rows and 0 <= nc < cols and board[nr * cols + nc] == mark:
                count += 1
            else:
                break
        if count >= inarow:
            return True
    return False


# test: place three 1s in the bottom row and check if a fourth would win
test_board = [0] * 42
test_board[35] = 1  # row 5, col 0
test_board[36] = 1  # row 5, col 1
test_board[37] = 1  # row 5, col 2
print('Win if drop at col 3:', is_win(test_board, 3, 1, 6, 7, 4))  # True
print('Win if drop at col 4:', is_win(test_board, 4, 1, 6, 7, 4))  # False

### 5.3 Board Evaluation Heuristic

When the minimax search reaches its depth limit, it cannot see who
wins. Instead, it uses a heuristic to estimate how good the board
looks for the current player. The heuristic slides a window of 4
cells across every row, column, and diagonal and scores each window:

| Window contents | Score |
|---|---|
| 4 own pieces | +1,000,000 (win) |
| 3 own + 1 empty | +50 |
| 2 own + 2 empty | +10 |
| 3 opponent + 1 empty | -80 |
| 2 opponent + 2 empty | -12 |
| 4 opponent | -1,000,000 (loss) |

The opponent threat score (-80) is deliberately higher in magnitude
than the own three-in-a-row score (+50). This makes the agent
prioritize blocking over attacking, which is critical in Connect
Four because a missed block often leads to an immediate loss.

An additional center column bonus rewards placing pieces in the
middle, since the center column participates in more possible
four-in-a-row combinations than any edge column.

In [ ]:
def evaluate_board(board, mark, rows, cols, inarow):
    """Score the board from the perspective of the given mark.
    Higher scores indicate a more favorable position."""
    opp_mark = 3 - mark  # if mark is 1 then opp is 2, and vice versa
    score = 0

    # bonus for pieces in the center column (column 3 on a 7-wide board)
    center_col = cols // 2
    center_count = sum(1 for r in range(rows) if board[r * cols + center_col] == mark)
    score += center_count * 6

    # score every window of 4 cells in all four directions

    # horizontal windows
    for r in range(rows):
        for c in range(cols - (inarow - 1)):
            window = [board[r * cols + c + i] for i in range(inarow)]
            score += score_window(window, mark, opp_mark)

    # vertical windows
    for r in range(rows - (inarow - 1)):
        for c in range(cols):
            window = [board[(r + i) * cols + c] for i in range(inarow)]
            score += score_window(window, mark, opp_mark)

    # positive diagonal windows (top-left to bottom-right)
    for r in range(rows - (inarow - 1)):
        for c in range(cols - (inarow - 1)):
            window = [board[(r + i) * cols + (c + i)] for i in range(inarow)]
            score += score_window(window, mark, opp_mark)

    # negative diagonal windows (bottom-left to top-right)
    for r in range(inarow - 1, rows):
        for c in range(cols - (inarow - 1)):
            window = [board[(r - i) * cols + (c + i)] for i in range(inarow)]
            score += score_window(window, mark, opp_mark)

    return score


def score_window(window, mark, opp_mark):
    """Score a single window of 4 cells."""
    own   = window.count(mark)
    opp   = window.count(opp_mark)
    empty = window.count(0)

    # a window with both players' pieces cannot form a four-in-a-row for either
    if own > 0 and opp > 0:
        return 0

    if own == 4:
        return 1_000_000
    if own == 3 and empty == 1:
        return 50
    if own == 2 and empty == 2:
        return 10

    if opp == 4:
        return -1_000_000
    if opp == 3 and empty == 1:
        return -80
    if opp == 2 and empty == 2:
        return -12

    return 0


# test on empty board and after a few moves
empty_board = [0] * 42
print('Empty board score:', evaluate_board(empty_board, 1, 6, 7, 4))

# place three 1s in the bottom center
test_board = [0] * 42
test_board[38] = 1  # row 5, col 3
test_board[39] = 1  # row 5, col 4
test_board[40] = 1  # row 5, col 5
print('Three in a row score:', evaluate_board(test_board, 1, 6, 7, 4))

### 5.4 Minimax with Alpha-Beta Pruning

Minimax explores the game tree by alternating between maximizing
(our turn) and minimizing (opponent's turn) layers. At each node
it tries every valid column, simulates the move, and recurses.

Alpha-beta pruning adds two bounds:
- **alpha:** the best score the maximizing player can guarantee so far
- **beta:** the best score the minimizing player can guarantee so far

When alpha >= beta, it means the current branch cannot produce a
result better than what is already available, so it is pruned.
This dramatically reduces the number of nodes explored, especially
when moves are ordered well.

The search also checks for terminal states (win, loss, draw) before
recursing, returning the true outcome directly rather than an
estimate.

In [ ]:
def minimax(board, mark, depth, alpha, beta, is_maximizing, rows, cols, inarow):
    """Minimax search with alpha-beta pruning.
    Returns the heuristic score of the board from the perspective of mark."""
    opp_mark = 3 - mark
    valid_cols = get_valid_columns(board, cols)

    # terminal check: no valid moves means the board is full (draw)
    if not valid_cols:
        return 0

    # depth limit reached: return heuristic evaluation
    if depth == 0:
        return evaluate_board(board, mark, rows, cols, inarow)

    # order columns from center outward for better pruning
    center = cols // 2
    valid_cols.sort(key=lambda c: abs(c - center))

    if is_maximizing:
        max_score = -float('inf')
        for col in valid_cols:
            # check if this move wins immediately
            if is_win(board, col, mark, rows, cols, inarow):
                return 1_000_000 + depth  # prefer faster wins
            new_board = drop_checker(board, col, mark, rows, cols)
            score = minimax(new_board, mark, depth - 1, alpha, beta, False, rows, cols, inarow)
            max_score = max(max_score, score)
            alpha = max(alpha, score)
            if alpha >= beta:
                break  # beta cutoff
        return max_score
    else:
        min_score = float('inf')
        for col in valid_cols:
            # check if opponent wins immediately
            if is_win(board, col, opp_mark, rows, cols, inarow):
                return -1_000_000 - depth  # opponent wins
            new_board = drop_checker(board, col, opp_mark, rows, cols)
            score = minimax(new_board, mark, depth - 1, alpha, beta, True, rows, cols, inarow)
            min_score = min(min_score, score)
            beta = min(beta, score)
            if alpha >= beta:
                break  # alpha cutoff
        return min_score


print('Minimax function defined.')

### 5.5 Assembling the Final Agent

The top-level agent function ties everything together. On each turn
it:

1. Checks if any column gives an immediate win and takes it.
2. Checks if the opponent can win next turn and blocks it.
3. Otherwise, runs minimax at depth 5 on each valid column and
   picks the column with the highest score.

Steps 1 and 2 are checked before the search because they are
trivial to compute and catch the most critical situations without
wasting time on a full tree search.

In [ ]:
def agent(observation, configuration):
    """ConnectX agent using minimax with alpha-beta pruning."""
    import random

    board  = observation.board
    mark   = observation.mark
    cols   = configuration.columns
    rows   = configuration.rows
    inarow = configuration.inarow
    opp_mark = 3 - mark

    # all helper functions are defined inside the agent so the submission file
    # is fully self-contained and has no external dependencies

    def _get_valid(board):
        """Return a list of columns that still have at least one empty cell."""
        return [c for c in range(cols) if board[c] == 0]

    def _drop(board, col, mk):
        """Simulate dropping a checker into the given column and return the resulting board."""
        b = list(board)  # create a copy so the original board is not modified
        for r in range(rows - 1, -1, -1):
            if b[r * cols + col] == 0:
                b[r * cols + col] = mk
                return b
        return b

    def _is_win(board, col, mk):
        """Check whether dropping a checker into the given column would create four in a row."""
        # find the row where the checker would land
        row = -1
        for r in range(rows - 1, -1, -1):
            if board[r * cols + col] == 0:
                row = r
                break
        if row == -1:
            return False  # the column is completely full
        # check all four directions: horizontal, vertical, and both diagonals
        for dr, dc in [(0,1),(1,0),(1,1),(1,-1)]:
            cnt = 1  # count the piece being placed
            # count consecutive matching pieces in the positive direction
            for i in range(1, inarow):
                nr, nc = row + dr*i, col + dc*i
                if 0 <= nr < rows and 0 <= nc < cols and board[nr*cols+nc] == mk:
                    cnt += 1
                else:
                    break
            # count consecutive matching pieces in the negative direction
            for i in range(1, inarow):
                nr, nc = row - dr*i, col - dc*i
                if 0 <= nr < rows and 0 <= nc < cols and board[nr*cols+nc] == mk:
                    cnt += 1
                else:
                    break
            if cnt >= inarow:
                return True
        return False

    def _score_window(window):
        """Assign a numeric score to a window of four consecutive cells."""
        own   = window.count(mark)
        opp   = window.count(opp_mark)
        empty = window.count(0)
        # if both players have pieces in this window, neither can complete four in a row
        if own > 0 and opp > 0:
            return 0
        if own == 4:
            return 1_000_000
        if own == 3 and empty == 1:
            return 50
        if own == 2 and empty == 2:
            return 10
        if opp == 4:
            return -1_000_000
        if opp == 3 and empty == 1:
            return -80
        if opp == 2 and empty == 2:
            return -12
        return 0

    def _evaluate(board):
        """Evaluate the entire board and return a numeric score from our perspective."""
        score = 0
        center = cols // 2
        # award bonus points for pieces in the center column
        score += sum(1 for r in range(rows) if board[r*cols+center] == mark) * 6
        # scan all horizontal windows of four consecutive cells
        for r in range(rows):
            for c in range(cols - (inarow-1)):
                w = [board[r*cols+c+i] for i in range(inarow)]
                score += _score_window(w)
        # scan all vertical windows
        for r in range(rows - (inarow-1)):
            for c in range(cols):
                w = [board[(r+i)*cols+c] for i in range(inarow)]
                score += _score_window(w)
        # scan all positive diagonal windows (top-left to bottom-right)
        for r in range(rows - (inarow-1)):
            for c in range(cols - (inarow-1)):
                w = [board[(r+i)*cols+(c+i)] for i in range(inarow)]
                score += _score_window(w)
        # scan all negative diagonal windows (bottom-left to top-right)
        for r in range(inarow-1, rows):
            for c in range(cols - (inarow-1)):
                w = [board[(r-i)*cols+(c+i)] for i in range(inarow)]
                score += _score_window(w)
        return score

    def _minimax(board, depth, alpha, beta, maximizing):
        """Recursive minimax search with alpha-beta pruning."""
        valid = _get_valid(board)
        if not valid:
            return 0  # all cells filled, the game is a draw
        if depth == 0:
            return _evaluate(board)  # depth limit reached, use the heuristic
        # order columns from center outward so the strongest moves are searched first
        center = cols // 2
        valid.sort(key=lambda c: abs(c - center))
        if maximizing:
            best = -float('inf')
            for col in valid:
                if _is_win(board, col, mark):
                    return 1_000_000 + depth
                nb = _drop(board, col, mark)
                s = _minimax(nb, depth-1, alpha, beta, False)
                best = max(best, s)
                alpha = max(alpha, s)
                if alpha >= beta:
                    break
            return best
        else:
            best = float('inf')
            for col in valid:
                if _is_win(board, col, opp_mark):
                    return -1_000_000 - depth
                nb = _drop(board, col, opp_mark)
                s = _minimax(nb, depth-1, alpha, beta, True)
                best = min(best, s)
                beta = min(beta, s)
                if alpha >= beta:
                    break
            return best

    # -- main decision logic starts here --

    valid_cols = _get_valid(board)

    # step 1: if any column produces an immediate win, take it without searching
    for col in valid_cols:
        if _is_win(board, col, mark):
            return col

    # step 2: if the opponent could win on their next turn, block that column
    for col in valid_cols:
        if _is_win(board, col, opp_mark):
            return col

    # step 3: run the minimax search on each valid column and choose the highest-scoring one
    search_depth = 6
    best_score = -float('inf')
    best_col = valid_cols[0]

    # search columns starting from the center outward to improve alpha-beta pruning
    center = cols // 2
    valid_cols.sort(key=lambda c: abs(c - center))

    for col in valid_cols:
        new_board = _drop(board, col, mark)
        score = _minimax(new_board, search_depth - 1, -float('inf'), float('inf'), False)
        if score > best_score:
            best_score = score
            best_col = col

    return best_col


print('Agent function defined.')

## 6. Local Testing

Before submitting, we test the agent locally against the built-in
random and negamax agents. The `evaluate` function runs multiple
episodes and returns the outcome of each game as [player1_reward,
player2_reward] where 1 = win, 0.5 = draw, 0 = loss.

In [ ]:
# run a single game between our agent and the built-in negamax agent
env.reset()
env.run([agent, 'negamax'])

# display the final board state as text
print(env.render(mode='ansi'))

In [ ]:
import numpy as np

def test_agent(agent1, agent2, n_rounds=50):
    """Run n_rounds games with agent1 going first half the time."""
    # agent1 goes first
    results_first = evaluate('connectx', [agent1, agent2], num_episodes=n_rounds // 2)
    # agent1 goes second
    results_second = evaluate('connectx', [agent2, agent1], num_episodes=n_rounds // 2)

    wins = sum(1 for r in results_first if r[0] == 1) + sum(1 for r in results_second if r[1] == 1)
    losses = sum(1 for r in results_first if r[1] == 1) + sum(1 for r in results_second if r[0] == 1)
    draws = n_rounds - wins - losses

    print(f'  Wins: {wins}/{n_rounds} ({100*wins/n_rounds:.0f}%)')
    print(f'  Losses: {losses}/{n_rounds} ({100*losses/n_rounds:.0f}%)')
    print(f'  Draws: {draws}/{n_rounds} ({100*draws/n_rounds:.0f}%)')
    return wins, losses, draws

In [ ]:
print('Agent vs Random:')
test_agent(agent, 'random', n_rounds=20)
print()
print('Agent vs Negamax:')
test_agent(agent, 'negamax', n_rounds=20)

## 7. Submission File

The competition requires a `.py` file containing the agent function.
We use `inspect.getsource` to extract the function source code and
write it to `submission.py`. This file is then available in the
notebook output for submission.

In [ ]:
import inspect
import os

# write the complete agent function source code to submission.py
with open('submission.py', 'w') as f:
    f.write(inspect.getsource(agent))

print('submission.py written successfully.')
print(f'File size: {os.path.getsize("submission.py")} bytes')

## 8. Validate Submission

The competition runs a validation episode where the submission plays
against itself. If this episode fails, the submission is rejected.
We replicate that check here to make sure everything works before
uploading.

In [ ]:
# load and validate the submission file directly path
env = make('connectx', debug=True)
env.run(['submission.py', 'submission.py'])

status_ok = env.state[0].status == env.state[1].status == 'DONE'
print('Validation:', 'PASSED' if status_ok else 'FAILED')

## 9. Summary

What we built:

1. A flat-list board representation for fast indexing without NumPy.
2. Directional win detection that only checks from the last placed piece.
3. A window-based board evaluation heuristic with center column bonus
   and asymmetric attack/defense weighting.
4. Minimax search with alpha-beta pruning and center-first move ordering.
5. Immediate win/block checks before entering the tree search.
6. A self-contained agent function written to `submission.py`.

---

**Citation:**
Adam, Addison Howard, and Bovard Doerschuk-Tiberi. Connect X.
https://kaggle.com/competitions/connectx, 2020. Kaggle.